# Cache-Augmented RAG Pipeline [Step 3 - Full Cache RAG with Graph]

> **MLCourse - Agentic AI - Cache RAG**

This notebook combines semantic caching with a full RAG pipeline into a
LangGraph graph. The agent checks the cache first: on a hit, it returns
the cached answer immediately. On a miss, it retrieves documents, generates
an answer, and stores the result in the cache. The graph visualization
shows the branching logic clearly.

In [1]:
import os
import time
import hashlib
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()

False

In [2]:
api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print("[GREEN] API key found")
else:
    print("[GREEN] No API key needed -- using local ChatOllama")

[GREEN] No API key needed -- using local ChatOllama


### 1. Load and Chunk the Document


In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_path = r"D:\projects\python\MLCourse\03_agentic_ai\data\alice.txt"
with open(text_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_text(raw_text)
print(f"Loaded alice.txt: {len(raw_text)} chars -> {len(chunks)} chunks")


### 2. Build the Vector Store


In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_texts(chunks, embeddings, collection_name="alice_full_cache_rag")
print(f"Vector store built with {vectorstore._collection.count()} vectors")

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


### 3. Initialize the LLM


In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1:8b", temperature=0)
print("LLM initialized:", llm.model)


### 4. Build the Semantic Cache


In [ ]:
# Combines embedding-based lookup with TTL and content-hash invalidation.

class SemanticCacheRAG:
    """Semantic cache with TTL and content-hash invalidation."""

    def __init__(self, embeddings_model, threshold=0.85, ttl_seconds=3600):
        self.embeddings_model = embeddings_model
        self.threshold = threshold
        self.ttl = ttl_seconds
        self.cache = []  # list of dicts
        self.content_hash = None

    def set_content_hash(self, text):
        self.content_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]

    def _get_embedding(self, text):
        return self.embeddings_model.embed_query(text)

    def _cosine_similarity(self, a, b):
        a, b = np.array(a), np.array(b)
        dot = np.dot(a, b)
        norm_a, norm_b = np.linalg.norm(a), np.linalg.norm(b)
        if norm_a == 0 or norm_b == 0:
            return 0.0
        return float(dot / (norm_a * norm_b))

    def lookup(self, query):
        """Return (entry, score, reason) or (None, best_score, reason)."""
        if not self.cache:
            return None, 0.0, "empty_cache"
        query_emb = self._get_embedding(query)
        best_score = -1.0
        best_entry = None
        for entry in self.cache:
            # Skip expired entries
            if time.time() - entry["timestamp"] > self.ttl:
                continue
            # Skip entries with mismatched content hash
            if entry.get("source_hash") != self.content_hash:
                continue
            score = self._cosine_similarity(query_emb, entry["embedding"])
            if score > best_score:
                best_score = score
                best_entry = entry
        if best_entry and best_score >= self.threshold:
            return best_entry, best_score, "hit"
        return None, best_score, "miss"

    def store(self, query, answer):
        self.cache.append({
            "query": query,
            "embedding": self._get_embedding(query),
            "answer": answer,
            "timestamp": time.time(),
            "source_hash": self.content_hash,
        })

    def size(self):
        return len(self.cache)

    def clear(self):
        self.cache = []

cache = SemanticCacheRAG(embeddings, threshold=0.85, ttl_seconds=3600)
cache.set_content_hash(raw_text)
print(f"Semantic cache ready: threshold={cache.threshold}, TTL={cache.ttl}s")


### 5. Define the Graph State


In [ ]:
# The state tracks the full pipeline: query, cache lookup result,
# retrieved documents, generated answer, and metadata.

from typing import TypedDict, Literal

class CacheRAGState(TypedDict):
    query: str
    cache_hit: bool
    cache_score: float
    cache_answer: str
    retrieved_context: str
    answer: str
    from_cache: bool

print("CacheRAGState defined.")


### 6. Define Prompts


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the user's question using ONLY the provided context. "
     "If the context does not contain enough information, say so. "
     "Be concise and accurate."),
    ("user", "Context:\n{context}\n\nQuestion: {query}")
])

print("Prompts defined.")


### 7. Build Graph Nodes


In [ ]:
def check_cache(state: CacheRAGState) -> dict:
    """Node: Check semantic cache for a matching query."""
    query = state["query"]
    entry, score, reason = cache.lookup(query)
    if entry:
        print(f"  [CACHE] HIT (sim={score:.4f}) -- '{entry['query'][:50]}...'")
        return {
            "cache_hit": True,
            "cache_score": score,
            "cache_answer": entry["answer"],
        }
    print(f"  [CACHE] MISS (best_sim={score:.4f}, reason={reason})")
    return {"cache_hit": False, "cache_score": score, "cache_answer": ""}

def retrieve_documents(state: CacheRAGState) -> dict:
    """Node: Retrieve relevant documents from the vector store."""
    query = state["query"]
    docs = retriever.invoke(query)
    context = "\n\n---\n\n".join([d.page_content for d in docs])
    print(f"  [RETRIEVE] Got {len(docs)} docs ({len(context)} chars)")
    return {"retrieved_context": context}

def generate_answer(state: CacheRAGState) -> dict:
    """Node: Generate answer using the LLM with retrieved context."""
    response = (rag_prompt | llm).invoke({
        "context": state["retrieved_context"],
        "query": state["query"]
    })
    print(f"  [GENERATE] {response.content[:80]}...")
    return {"answer": response.content}

def cache_answer(state: CacheRAGState) -> dict:
    """Node: Store the generated answer in the semantic cache."""
    cache.store(state["query"], state["answer"])
    print(f"  [CACHE-STORE] Cache size: {cache.size()}")
    return {"from_cache": False}

def return_cached(state: CacheRAGState) -> dict:
    """Node: Return the cached answer directly."""
    print(f"  [RETURN-CACHED] Using cached answer (sim={state['cache_score']:.4f})")
    return {
        "answer": state["cache_answer"],
        "from_cache": True,
    }

print("All graph nodes defined.")


### 8. Define Routing Logic


In [ ]:
def route_after_cache_check(state: CacheRAGState) -> Literal["return_cached", "retrieve_documents"]:
    """Route based on cache lookup result."""
    if state["cache_hit"]:
        return "return_cached"
    return "retrieve_documents"

print("Routing logic defined.")


### 9. Build the LangGraph


In [ ]:
from langgraph.graph import StateGraph, START, END

graph_builder = StateGraph(CacheRAGState)

# Add nodes
graph_builder.add_node("check_cache", check_cache)
graph_builder.add_node("return_cached", return_cached)
graph_builder.add_node("retrieve_documents", retrieve_documents)
graph_builder.add_node("generate_answer", generate_answer)
graph_builder.add_node("cache_answer", cache_answer)

# Entry edge
graph_builder.add_edge(START, "check_cache")

# Conditional routing after cache check
graph_builder.add_conditional_edges(
    "check_cache",
    route_after_cache_check,
    {
        "return_cached": "return_cached",
        "retrieve_documents": "retrieve_documents",
    }
)

# Cache hit path: return directly
graph_builder.add_edge("return_cached", END)

# Cache miss path: retrieve -> generate -> store -> end
graph_builder.add_edge("retrieve_documents", "generate_answer")
graph_builder.add_edge("generate_answer", "cache_answer")
graph_builder.add_edge("cache_answer", END)

graph = graph_builder.compile()
print("Cache-augmented RAG graph compiled.")


### 10. Visualize the Graph


In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Could not render graph image: {e}")
    print("Graph structure:")
    print("  START -> check_cache")
    print("  check_cache -> [hit] -> return_cached -> END")
    print("  check_cache -> [miss] -> retrieve_documents -> generate_answer -> cache_answer -> END")


### 11. Test: First Query (Cache Miss)


In [ ]:
print("=" * 60)
print("TEST 1: First query -- cache miss, full pipeline")
print("=" * 60)
result = graph.invoke({
    "query": "Who does Alice meet at the tea party?",
    "cache_hit": False,
    "cache_score": 0.0,
    "cache_answer": "",
    "retrieved_context": "",
    "answer": "",
    "from_cache": False,
})
print(f"\nAnswer: {result['answer'][:200]}")
print(f"From cache: {result['from_cache']}")


### 12. Test: Same Query Again (Cache Hit)


In [ ]:
print("\n" + "=" * 60)
print("TEST 2: Same query -- cache hit, instant return")
print("=" * 60)
result = graph.invoke({
    "query": "Who does Alice meet at the tea party?",
    "cache_hit": False,
    "cache_score": 0.0,
    "cache_answer": "",
    "retrieved_context": "",
    "answer": "",
    "from_cache": False,
})
print(f"\nAnswer: {result['answer'][:200]}")
print(f"From cache: {result['from_cache']}")


### 13. Test: Similar Query (Should Hit Cache)


In [ ]:
print("\n" + "=" * 60)
print("TEST 3: Similar phrasing -- should hit cache")
print("=" * 60)
result = graph.invoke({
    "query": "Which characters does Alice encounter at the Mad Hatter's tea party?",
    "cache_hit": False,
    "cache_score": 0.0,
    "cache_answer": "",
    "retrieved_context": "",
    "answer": "",
    "from_cache": False,
})
print(f"\nAnswer: {result['answer'][:200]}")
print(f"From cache: {result['from_cache']}")


### 14. Test: New Topic (Cache Miss)


In [ ]:
print("\n" + "=" * 60)
print("TEST 4: New topic -- cache miss")
print("=" * 60)
result = graph.invoke({
    "query": "What is the Queen of Hearts' favorite punishment?",
    "cache_hit": False,
    "cache_score": 0.0,
    "cache_answer": "",
    "retrieved_context": "",
    "answer": "",
    "from_cache": False,
})
print(f"\nAnswer: {result['answer'][:200]}")
print(f"From cache: {result['from_cache']}")


### 15. Test: Another New Topic


In [ ]:
print("\n" + "=" * 60)
print("TEST 5: Another new topic -- cache miss")
print("=" * 60)
result = graph.invoke({
    "query": "What games does the Queen of Hearts play?",
    "cache_hit": False,
    "cache_score": 0.0,
    "cache_answer": "",
    "retrieved_context": "",
    "answer": "",
    "from_cache": False,
})
print(f"\nAnswer: {result['answer'][:200]}")
print(f"From cache: {result['from_cache']}")


### 16. Test: Repeat New Topic (Should Now Hit Cache)


In [ ]:
print("\n" + "=" * 60)
print("TEST 6: Repeat of TEST 5 -- should hit cache now")
print("=" * 60)
result = graph.invoke({
    "query": "What games does the Queen of Hearts play?",
    "cache_hit": False,
    "cache_score": 0.0,
    "cache_answer": "",
    "retrieved_context": "",
    "answer": "",
    "from_cache": False,
})
print(f"\nAnswer: {result['answer'][:200]}")
print(f"From cache: {result['from_cache']}")


### 17. Cache Performance Summary


In [ ]:
print("\n" + "=" * 60)
print("CACHE PERFORMANCE SUMMARY")
print("=" * 60)
print(f"Total cache entries: {cache.size()}")
print(f"Cache threshold: {cache.threshold}")
print(f"Cache TTL: {cache.ttl}s")
print(f"Content hash: {cache.content_hash}")
print()

# Simulate hit/miss tracking
stats = {"hits": 0, "misses": 0}
test_queries = [
    "Who does Alice meet at the tea party?",
    "Which characters does Alice encounter at the Mad Hatter's tea party?",
    "What is the Queen of Hearts' favorite punishment?",
    "Tell me about the Cheshire Cat",
    "What games does the Queen of Hearts play?",
    "Describe the White Rabbit",
]
for q in test_queries:
    _, _, reason = cache.lookup(q)
    if reason == "hit":
        stats["hits"] += 1
    else:
        stats["misses"] += 1

total = stats["hits"] + stats["misses"]
print(f"Test queries: {total}")
print(f"Cache hits: {stats['hits']} ({100*stats['hits']/total:.0f}%)")
print(f"Cache misses: {stats['misses']} ({100*stats['misses']/total:.0f}%)")


### 18. Inspect Graph Structure


In [ ]:
print("=== Graph Structure ===")
g = graph.get_graph()
print(f"Nodes: {list(g.nodes.keys())}")
print("Edges:")
for edge in g.edges:
    print(f"  {edge.source} -> {edge.target}")


### Summary
